# Climate Data – A hands-on python course
Author: Pedro Herrera Lormendez (pedrolormendez@gmail.com)

**Updated 2025:** Major migration to new ECMWF datastores client with enhanced job management and async capabilities

## Climate Data Store (CDS) - Modern Data Access

The **Climate Data Store (CDS)** of Copernicus, operated by the European Centre for Medium-Range Weather Forecasts (ECMWF), is a comprehensive source for climate data. It is a key component of the **Copernicus Climate Change Service (C3S)**.

**Website:** https://cds.climate.copernicus.eu/

### Key datasets of interest:
* [ERA5 Reanalysis hourly surface data (single levels)](https://cds.climate.copernicus.eu/cdsapp#!/dataset/reanalysis-era5-single-levels?tab=overview)
* [ERA5 Reanalysis hourly pressure levels data](https://cds.climate.copernicus.eu/cdsapp#!/dataset/reanalysis-era5-pressure-levels?tab=overview)
* [ERA5 Reanalysis monthly surface data](https://cds.climate.copernicus.eu/cdsapp#!/dataset/reanalysis-era5-single-levels-monthly-means?tab=overview)
* [ERA5 Reanalysis monthly pressure levels data](https://cds.climate.copernicus.eu/cdsapp#!/dataset/reanalysis-era5-pressure-levels-monthly-means?tab=overview)
* [CMIP6 Global Climate Models](https://cds.climate.copernicus.eu/cdsapp#!/dataset/projections-cmip6?tab=overview)
* [Climate projections IPCC AR6 Interactive Atlas](https://cds.climate.copernicus.eu/cdsapp#!/dataset/projections-climate-atlas?tab=form)

### Two ways to access data:
1. **Web interface:** Direct download in .grib or .nc format
2. **API (Programmatic access):** Using Python (recommended for reproducibility and automation)

---
## ⚠️ IMPORTANT: API Migration in 2024-2025

ECMWF has introduced a **new unified client** for accessing their data stores:

### New Client (Recommended): `ecmwf-datastores-client`
* **Package:** `ecmwf-datastores-client`
* **Supports:** CDS, ADS (Atmosphere Data Store), and future stores
* **Features:**
  * Unified interface across multiple data stores
  * Enhanced job tracking and management
  * Asynchronous data retrieval
  * Better error handling and validation
  * Result inspection before download
  * Support for multiple authentication methods

### Legacy Client (Still supported): `cdsapi`
* **Package:** `cdsapi`
* **Status:** Maintained but not receiving new features
* **Will work:** For existing CDS datasets

**This notebook demonstrates BOTH approaches**, with emphasis on the new client.

---

## Setup and Authentication

### Step 1: Register for an account
1. Go to https://cds.climate.copernicus.eu/
2. Create a free account
3. Verify your email

### Step 2: Get your API credentials
1. Log in to CDS
2. Go to your user page: https://cds.climate.copernicus.eu/user
3. Scroll down to "API key" section
4. Copy your UID and API key

### Step 3: Install the clients

In [ ]:
# Install the new ECMWF datastores client (recommended)
# !pip install ecmwf-datastores-client

# Install legacy cdsapi (if needed for comparison)
# !pip install cdsapi

### Step 4: Configure authentication

**For the new client (`ecmwf-datastores-client`):**

Create a configuration file at `~/.ecmwfapirc` or `~/.config/ecmwf/config.yaml`:

```yaml
# ~/.config/ecmwf/config.yaml
datastores:
  cds:
    url: https://cds.climate.copernicus.eu/api
    key: YOUR_UID:YOUR_API_KEY
```

**For legacy cdsapi:**

Create `~/.cdsapirc`:

```
url: https://cds.climate.copernicus.eu/api/v2
key: YOUR_UID:YOUR_API_KEY
```

**Security note:** Never commit API keys to version control!

---
## New ECMWF Datastores Client (Recommended)

### Advantages:
* **Unified interface** for CDS and ADS
* **Better job management** with tracking and cancellation
* **Async support** for non-blocking operations
* **Result inspection** before downloading
* **Enhanced error messages** and validation
* **Future-proof** architecture

In [ ]:
from ecmwf.datastores import Client
import xarray as xr
import os
from pathlib import Path
import time

### Initializing the client

In [ ]:
# Initialize the client (automatically reads from config)
try:
    client = Client()
    print("✓ Client initialized successfully")
    print(f"Available datastores: {client.datastores}")
except Exception as e:
    print(f"Error initializing client: {e}")
    print("\nMake sure you have:")
    print("1. Installed: pip install ecmwf-datastores-client")
    print("2. Created config file with your API credentials")
    print("3. Registered at https://cds.climate.copernicus.eu/")

### Exploring available collections

The new client allows you to programmatically discover available datasets (collections).

In [ ]:
# List all available collections in CDS
try:
    collections = client.get_collections(datastore='cds')
    print(f"Total collections available: {len(collections)}\n")
    
    # Display first 10 collections
    print("Sample collections:")
    print("-" * 80)
    for i, collection in enumerate(collections[:10]):
        print(f"{i+1}. {collection['id']}")
        print(f"   {collection.get('title', 'No title')}")
        print()
except Exception as e:
    print(f"Note: Collection listing may require authentication. Error: {e}")

### Getting collection details

Explore metadata and available parameters for a specific dataset.

In [ ]:
# Get details for ERA5 monthly single levels
collection_id = 'reanalysis-era5-single-levels-monthly-means'

try:
    collection_info = client.get_collection(collection_id, datastore='cds')
    
    print(f"Collection: {collection_info['id']}")
    print(f"Title: {collection_info.get('title', 'N/A')}")
    print(f"Description: {collection_info.get('description', 'N/A')[:200]}...")
    print(f"\nAvailable variables: {len(collection_info.get('variables', []))} variables")
    
    # Show some example variables
    if 'variables' in collection_info:
        print("\nExample variables:")
        for var in list(collection_info['variables'].keys())[:5]:
            print(f"  - {var}")
except Exception as e:
    print(f"Could not retrieve collection details: {e}")
    print("This is normal if you haven't set up authentication yet.")

---
## Data Retrieval with the New Client

### Example 1: Synchronous retrieval (simple, blocking)

This is the simplest approach - the script waits until data is ready and downloaded.

In [ ]:
# Define output directory
data_dir = Path('../data')
data_dir.mkdir(exist_ok=True)

# Define the request
request = {
    'product_type': 'monthly_averaged_reanalysis',
    'variable': '2m_temperature',
    'year': '2023',
    'month': [
        '01', '02', '03', '04', '05', '06',
        '07', '08', '09', '10', '11', '12'
    ],
    'time': '00:00',
    'format': 'netcdf',
    'area': [70, -20, 30, 40],  # North, West, South, East (Europe)
}

output_file = data_dir / 'era5_monthly_t2m_2023_europe.nc'

print("Starting synchronous data retrieval...")
print(f"Request: {request}")
print(f"Output: {output_file}")
print("\nNote: This may take several minutes. The job will be queued and processed.")

try:
    # Submit request and download (blocking)
    result = client.retrieve(
        collection_id='reanalysis-era5-single-levels-monthly-means',
        request=request,
        target=str(output_file)
    )
    print(f"\n✓ Download complete: {output_file}")
    print(f"File size: {output_file.stat().st_size / 1024 / 1024:.2f} MB")
except Exception as e:
    print(f"\nError during retrieval: {e}")
    print("\nPossible reasons:")
    print("1. API credentials not configured")
    print("2. Network connection issues")
    print("3. Request format error")
    print("4. Queue is busy (try again later)")

### Example 2: Asynchronous retrieval with job management

This approach gives you more control:
* Submit job without blocking
* Monitor progress
* Inspect results before downloading
* Cancel if needed

In [ ]:
# Submit the request asynchronously
request_async = {
    'product_type': 'monthly_averaged_reanalysis',
    'variable': ['2m_temperature', 'total_precipitation'],
    'year': '2023',
    'month': ['06', '07', '08'],  # Summer months
    'time': '00:00',
    'format': 'netcdf',
    'area': [60, -10, 35, 30],  # Europe
}

output_file_async = data_dir / 'era5_summer_2023.nc'

print("Submitting asynchronous request...")
print(f"Request: {request_async}\n")

try:
    # Submit without waiting for completion
    job = client.submit(
        collection_id='reanalysis-era5-single-levels-monthly-means',
        request=request_async
    )
    
    print(f"✓ Job submitted successfully")
    print(f"Job ID: {job.job_id}")
    print(f"Status: {job.status}")
    
    # Monitor job progress
    print("\nMonitoring job progress...")
    while job.status in ['queued', 'running']:
        print(f"  Status: {job.status} | {time.strftime('%H:%M:%S')}")
        time.sleep(10)  # Check every 10 seconds
        job.update()  # Refresh job status
    
    if job.status == 'completed':
        print(f"\n✓ Job completed successfully!")
        
        # Inspect results before downloading
        result_info = job.result()
        print(f"\nResult information:")
        print(f"  Size: {result_info.get('size', 'unknown')}")
        print(f"  Location: {result_info.get('location', 'unknown')}")
        
        # Download the result
        print(f"\nDownloading to: {output_file_async}")
        job.download(str(output_file_async))
        print(f"✓ Download complete")
    else:
        print(f"\n✗ Job failed with status: {job.status}")
        print(f"Error: {job.error}")
        
except Exception as e:
    print(f"Error: {e}")

### Example 3: Managing multiple jobs

You can submit multiple requests and manage them collectively.

In [ ]:
# List all your active jobs
try:
    jobs = client.get_jobs(datastore='cds')
    
    print(f"Active jobs: {len(jobs)}\n")
    print("=" * 80)
    
    for job in jobs[:10]:  # Show up to 10 most recent
        print(f"Job ID: {job.job_id}")
        print(f"  Collection: {job.collection_id}")
        print(f"  Status: {job.status}")
        print(f"  Created: {job.created_at}")
        if job.status == 'completed':
            print(f"  Result size: {job.result().get('size', 'N/A')}")
        print()
        
except Exception as e:
    print(f"Could not retrieve jobs: {e}")

### Example 4: Retrieving results from existing jobs

If you have a job ID from a previous session, you can retrieve its results.

In [ ]:
# Example: Retrieve a job by ID
# Replace with your actual job ID
job_id = "your-job-id-here"

try:
    # Get remote job
    remote_job = client.get_remote(job_id, datastore='cds')
    
    print(f"Job ID: {remote_job.job_id}")
    print(f"Status: {remote_job.status}")
    
    if remote_job.status == 'completed':
        # Download the result
        output_path = data_dir / f'retrieved_{job_id}.nc'
        remote_job.download(str(output_path))
        print(f"✓ Downloaded to: {output_path}")
    else:
        print(f"Job not ready. Current status: {remote_job.status}")
        
except Exception as e:
    print(f"Could not retrieve job: {e}")
    print("This is normal if you don't have a valid job ID yet.")

---
## Error Handling Best Practices

Robust error handling is crucial for production workflows.

In [ ]:
def robust_cds_retrieval(collection_id, request, output_file, max_retries=3):
    """
    Robust data retrieval with error handling and retries.
    
    Parameters:
    -----------
    collection_id : str
        CDS collection identifier
    request : dict
        Request parameters
    output_file : str or Path
        Output file path
    max_retries : int
        Maximum number of retry attempts
    
    Returns:
    --------
    bool : Success status
    """
    from ecmwf.datastores.exceptions import (
        DatastoreError,
        AuthenticationError,
        RequestError
    )
    
    client = Client()
    
    for attempt in range(max_retries):
        try:
            print(f"Attempt {attempt + 1}/{max_retries}...")
            
            # Submit job
            job = client.submit(collection_id=collection_id, request=request)
            print(f"Job submitted: {job.job_id}")
            
            # Wait for completion with timeout
            timeout = 3600  # 1 hour
            start_time = time.time()
            
            while job.status in ['queued', 'running']:
                if time.time() - start_time > timeout:
                    raise TimeoutError(f"Job exceeded timeout of {timeout}s")
                
                time.sleep(30)
                job.update()
            
            # Check completion
            if job.status == 'completed':
                job.download(str(output_file))
                print(f"✓ Success: {output_file}")
                return True
            else:
                print(f"✗ Job failed: {job.status}")
                print(f"Error message: {job.error}")
                raise RequestError(f"Job failed: {job.error}")
                
        except AuthenticationError as e:
            print(f"✗ Authentication error: {e}")
            print("Please check your API credentials.")
            return False
            
        except RequestError as e:
            print(f"✗ Request error: {e}")
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt * 60  # Exponential backoff
                print(f"Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                print("Max retries reached. Please check your request parameters.")
                return False
                
        except TimeoutError as e:
            print(f"✗ Timeout: {e}")
            if attempt < max_retries - 1:
                print("Retrying...")
            else:
                print("Max retries reached. The queue may be very busy.")
                return False
                
        except Exception as e:
            print(f"✗ Unexpected error: {type(e).__name__}: {e}")
            if attempt < max_retries - 1:
                time.sleep(60)
            else:
                return False
    
    return False

# Example usage
print("Testing robust retrieval function...\n")
success = robust_cds_retrieval(
    collection_id='reanalysis-era5-single-levels-monthly-means',
    request={
        'product_type': 'monthly_averaged_reanalysis',
        'variable': '2m_temperature',
        'year': '2023',
        'month': '01',
        'time': '00:00',
        'format': 'netcdf',
    },
    output_file=data_dir / 'era5_test_robust.nc',
    max_retries=2
)

if success:
    print("\n✓ Robust retrieval completed successfully")
else:
    print("\n✗ Robust retrieval failed after all attempts")

---
## Legacy CDS API (cdsapi) - For Reference

The original `cdsapi` still works but is not receiving new features. Here's how to use it for comparison.

In [ ]:
# Legacy approach (commented out - uncomment to use)
'''
import cdsapi

# Initialize legacy client
c = cdsapi.Client()

# Submit request (blocking - waits for completion)
c.retrieve(
    'reanalysis-era5-single-levels-monthly-means',
    {
        'product_type': 'monthly_averaged_reanalysis',
        'variable': '2m_temperature',
        'year': '2023',
        'month': [
            '01', '02', '03', '04', '05', '06',
            '07', '08', '09', '10', '11', '12',
        ],
        'time': '00:00',
        'format': 'netcdf',
    },
    '../data/era5_monthly_t2m_2023_legacy.nc'
)

print("Legacy download complete")
'''

print("Legacy cdsapi example (commented out)")
print("Differences from new client:")
print("  - Simpler API but less flexible")
print("  - No job management capabilities")
print("  - No async support")
print("  - Blocking operations only")
print("  - Cannot inspect results before download")

### Feature Comparison

| Feature | New Client (`ecmwf-datastores-client`) | Legacy (`cdsapi`) |
|---------|--------------------------------------|------------------|
| **Async support** | ✅ Yes | ❌ No |
| **Job management** | ✅ Yes | ❌ No |
| **Multiple datastores** | ✅ CDS, ADS, more | ❌ CDS only |
| **Result inspection** | ✅ Yes | ❌ No |
| **Job cancellation** | ✅ Yes | ❌ No |
| **Collection discovery** | ✅ Yes | ❌ No |
| **Better error handling** | ✅ Yes | ⚠️ Basic |
| **Progress monitoring** | ✅ Yes | ⚠️ Limited |
| **Still maintained** | ✅ Yes | ✅ Yes (maintenance mode) |

**Recommendation:** Use the new client for all new projects.

---
## Data Validation After Download

Always validate downloaded data before analysis.

In [ ]:
def validate_downloaded_data(filepath):
    """
    Validate a downloaded NetCDF file.
    
    Checks:
    - File exists and is readable
    - File size is reasonable
    - NetCDF structure is valid
    - Required variables are present
    - Coordinate ranges are plausible
    - Data values are within physical bounds
    """
    import xarray as xr
    from pathlib import Path
    
    filepath = Path(filepath)
    
    print(f"Validating: {filepath}")
    print("=" * 60)
    
    # Check 1: File exists
    if not filepath.exists():
        print("✗ File does not exist")
        return False
    print(f"✓ File exists: {filepath.name}")
    
    # Check 2: File size
    size_mb = filepath.stat().st_size / 1024 / 1024
    print(f"✓ File size: {size_mb:.2f} MB")
    if size_mb < 0.01:
        print("⚠ Warning: File size is very small, may be corrupted")
    
    # Check 3: Can open as NetCDF
    try:
        ds = xr.open_dataset(filepath)
        print("✓ Valid NetCDF file")
    except Exception as e:
        print(f"✗ Cannot open as NetCDF: {e}")
        return False
    
    # Check 4: Metadata
    print(f"\nDimensions: {dict(ds.dims)}")
    print(f"Variables: {list(ds.data_vars)}")
    print(f"Coordinates: {list(ds.coords)}")
    
    # Check 5: Coordinate ranges
    if 'latitude' in ds.coords or 'lat' in ds.coords:
        lat = ds.latitude if 'latitude' in ds.coords else ds.lat
        print(f"\nLatitude range: {float(lat.min()):.2f} to {float(lat.max()):.2f}°")
        if lat.min() < -90 or lat.max() > 90:
            print("⚠ Warning: Latitude values outside valid range [-90, 90]")
    
    if 'longitude' in ds.coords or 'lon' in ds.coords:
        lon = ds.longitude if 'longitude' in ds.coords else ds.lon
        print(f"Longitude range: {float(lon.min()):.2f} to {float(lon.max()):.2f}°")
    
    if 'time' in ds.coords:
        print(f"Time range: {ds.time.values[0]} to {ds.time.values[-1]}")
        print(f"Number of time steps: {len(ds.time)}")
    
    # Check 6: Data values
    for var in ds.data_vars:
        data = ds[var]
        print(f"\nVariable: {var}")
        print(f"  Shape: {data.shape}")
        print(f"  Units: {data.attrs.get('units', 'Not specified')}")
        print(f"  Min: {float(data.min()):.2f}")
        print(f"  Max: {float(data.max()):.2f}")
        print(f"  Mean: {float(data.mean()):.2f}")
        
        # Physical plausibility checks for common variables
        if 't2m' in var or '2m_temperature' in var:
            # Temperature should be in reasonable range
            if data.attrs.get('units') == 'K':
                if data.min() < 180 or data.max() > 330:
                    print("  ⚠ Warning: Temperature values outside typical range")
                else:
                    print("  ✓ Temperature values in plausible range")
    
    ds.close()
    print("\n" + "=" * 60)
    print("✓ Validation complete")
    return True

# Example: Validate a downloaded file
test_file = data_dir / 'era5_monthly_t2m_2023_europe.nc'
if test_file.exists():
    validate_downloaded_data(test_file)
else:
    print(f"File not found: {test_file}")
    print("Run the retrieval cells above first.")

---
## Practice Time

<div style="background-color:lightgreen; padding:15px">
<b>Exercise: Download and analyze ERA5 data</b><br><br>

Using the new ECMWF datastores client:
<ol>
    <li>Download ERA5 monthly mean 2m temperature for your region of interest for 2020-2023</li>
    <li>Use asynchronous retrieval with job monitoring</li>
    <li>Validate the downloaded data</li>
    <li>Load with xarray and compute the annual mean temperature</li>
    <li>Create a simple plot showing the temperature trend</li>
</ol>

<b>Bonus challenges:</b>
<ul>
    <li>Download multiple variables (temperature and precipitation)</li>
    <li>Implement error handling with retries</li>
    <li>Compare file sizes for different spatial extents</li>
</ul>
</div>

In [ ]:
# Your code goes here
# Hint: Use the examples above as templates

# Step 1: Define your request
# ...

# Step 2: Submit asynchronously
# ...

# Step 3: Monitor and download
# ...

# Step 4: Validate
# ...

# Step 5: Analyze
# ...

---
## Other Important Data Sources

### ESGF (Earth System Grid Federation)
* **Purpose:** CMIP6 climate model data
* **Access:** Web interface or esgf-pyclient
* **Website:** https://esgf-node.llnl.gov/

### OPeNDAP (Open-source Project for a Network Data Access Protocol)
* **Purpose:** Remote data access without downloading
* **Access:** Direct URLs with xarray
* **Advantage:** Subset data server-side

**Example OPeNDAP access:**

In [ ]:
# Example: Access data via OPeNDAP (no download required)
try:
    # NOAA PSL OPeNDAP server example
    opendap_url = "https://psl.noaa.gov/thredds/dodsC/Datasets/ncep.reanalysis/surface/air.sig995.2023.nc"
    
    # Open directly (lazy loading)
    ds_remote = xr.open_dataset(opendap_url)
    
    print("✓ Connected to OPeNDAP server")
    print(f"Dataset: {ds_remote}")
    
    # Select subset (only this data is transferred)
    subset = ds_remote.sel(time='2023-07-15', lat=slice(60, 30), lon=slice(-10, 40))
    
    print(f"\nSubset shape: {subset.air.shape}")
    print("Only the selected subset is downloaded!")
    
except Exception as e:
    print(f"Could not access OPeNDAP server: {e}")
    print("This may be due to network issues or server availability.")

---
## Summary and Best Practices

### Key Takeaways:

1. **Use the new ECMWF datastores client** (`ecmwf-datastores-client`) for all new projects
   * More features, better job management, future-proof

2. **Asynchronous retrieval is powerful**
   * Don't block your workflow waiting for data
   * Monitor multiple jobs simultaneously
   * Inspect results before downloading

3. **Always implement error handling**
   * Network issues are common
   * Queue delays happen
   * Use retries with exponential backoff

4. **Validate downloaded data**
   * Check file integrity
   * Verify coordinates and metadata
   * Test physical plausibility

5. **Optimize your requests**
   * Request only needed variables
   * Subset spatially when possible
   * Use monthly means instead of hourly when appropriate
   * Consider NetCDF compression

### Request Optimization Tips:

```python
# ❌ Bad: Requesting entire globe when only need Europe
request = {
    'variable': 'all',  # All variables
    'year': range(1950, 2024),  # 74 years
    'month': range(1, 13),  # All months
    # No area specified = global
}

# ✅ Good: Specific, targeted request
request = {
    'variable': '2m_temperature',  # Only what you need
    'year': '2023',  # Specific year
    'month': ['06', '07', '08'],  # Summer only
    'area': [60, -10, 35, 30],  # Europe only
    'format': 'netcdf',  # Efficient format
}
```

### Additional Resources:

* **New client documentation:** https://datastores.readthedocs.io/
* **CDS documentation:** https://cds.climate.copernicus.eu/api-how-to
* **ERA5 documentation:** https://confluence.ecmwf.int/display/CKB/ERA5
* **Forum:** https://forum.ecmwf.int/

### Next Steps:
* **Notebook 4:** Computing climatologies and anomalies with WMO standards
* **Notebook 5:** Analyzing CMIP6 future climate projections
* **Notebook 6:** Climate attribution and extreme event analysis